In [ ]:
%matplotlib inline

import autoroot
import autorootcwd
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from pathlib import Path
from dataclasses import dataclass

In [ ]:
RESULTS_DIR = Path("artifacts/results/social/seed_1/medium")

In [ ]:
@dataclass
class CompositeSR:
    total: float
    random: float
    parallel: float
    circular: float


@dataclass
class ResultMetrics:
    sr: CompositeSR


@dataclass
class ControllerConfig:
    dynamics: str
    controller_type: str
    horizon: int
    n_samples: int
    stds_set: str
    result_dir: Path
    result_metrics: ResultMetrics
    presampler_type: str | None = None
    n_presamples: int | None = None

In [ ]:
def parse_controller(dir_path: str | Path) -> ControllerConfig:
    dir_path = Path(dir_path)
    dir_name = dir_path.name
    parts = dir_name.split("__")
    controller_name = parts[0]
    assert controller_name in ("mppi", "logmppi", "sampling")
    dynamics_name = parts[1]
    assert dynamics_name in ("unicycle", "bicycle")
    horizon = parts[2]
    assert horizon.startswith("h_")
    horizon = int(horizon[len("h_"):])
    stds = parts[3]
    assert stds.startswith("stds_")
    stds = stds[len("stds_"):]
    samples = parts[4]
    assert samples.startswith("samples_")
    samples = samples[len("samples_"):]
    if samples != "None":
        samples = int(samples)
    else:
        samples = None
    if len(parts) > 5:
        presampler = parts[5]
        presamples = parts[6]
        assert presamples.startswith("presamples_")
        presamples = int(presamples[len("presamples_"):])
    else:
        presampler = None
        presamples = None
    
    with open(dir_path / "metrics.json", "r") as f:
        metrics = json.load(f)
    sr = CompositeSR(
        total=metrics["success_rate"]["total"],
        random=metrics["success_rate"]["random"],
        parallel=metrics["success_rate"]["parallel"],
        circular=metrics["success_rate"]["circular"],
    )
    metrics = ResultMetrics(sr=sr)

    result = ControllerConfig(
        dynamics=dynamics_name,
        controller_type=controller_name,
        horizon=horizon,
        n_samples=samples,
        stds_set=stds,
        result_dir=dir_path,
        result_metrics=metrics,
        presampler_type=presampler,
        n_presamples=presamples,
    )
    return result

In [ ]:
RESULTS_NO_SAMPLING = []
RESULTS_SAMPLING_ONLY = []
RESULTS_ALL = []
for controller_dir in RESULTS_DIR.iterdir():
    controller_config = parse_controller(controller_dir)
    if controller_config.presampler_type == "nn_cuniform_steer":
        continue
    if controller_config.controller_type != "sampling":
        RESULTS_NO_SAMPLING.append(controller_config)
    else:
        RESULTS_SAMPLING_ONLY.append(controller_config)
    RESULTS_ALL.append(controller_config)

# Results analysis

## Total SR change (%-points) from adding a pre-sampler

In [ ]:
rows = []
for r in RESULTS_NO_SAMPLING:
    rows.append({
        "dynamics": r.dynamics,
        "horizon": r.horizon,
        "controller": r.controller_type,
        "stds": r.stds_set,
        "n_samples": r.n_samples,
        "presampler": r.presampler_type if r.presampler_type else "none",
        "n_presamples": r.n_presamples if r.n_presamples else 0,
        "total_sr": r.result_metrics.sr.total,
    })
df = pd.DataFrame(rows)

BASE_KEY_COLS = ["dynamics", "horizon", "controller", "stds", "n_samples"]
CONTROLLER_ORDER = ["mppi", "logmppi"]
PRESAMPLER_ORDER = ["nf", "fm", "nf_steer", "fm_steer"]
PRESAMPLES_ORDER = [500, 1000, 2500]

baselines = df[df["presampler"] == "none"].set_index(BASE_KEY_COLS)["total_sr"]
variants = df[df["presampler"] != "none"].copy()
variants["baseline_sr"] = variants.set_index(BASE_KEY_COLS).index.map(baselines)
variants["delta_pct"] = (variants["total_sr"] - variants["baseline_sr"]) / variants["baseline_sr"] * 100


def _format_cell(sr, delta):
    if pd.isna(sr) or pd.isna(delta):
        return "—"
    return f"{sr:.3f} ({delta:+.1f}%)"


def _color_delta(val):
    if not isinstance(val, str) or val == "—":
        return ""
    try:
        pct_str = val.split("(")[1].rstrip("%)")
        delta = float(pct_str)
    except (IndexError, ValueError):
        return ""
    if delta > 0:
        intensity = min(abs(delta) / 20, 1.0)
        return f"background-color: rgba(76, 175, 80, {0.15 + 0.55 * intensity}); color: #1b5e20"
    if delta < 0:
        intensity = min(abs(delta) / 20, 1.0)
        return f"background-color: rgba(244, 67, 54, {0.15 + 0.55 * intensity}); color: #b71c1c"
    return "background-color: rgba(158, 158, 158, 0.1)"


def _border_styles(pivot):
    """Build per-cell border styles for visual group separation."""
    styles = pd.DataFrame("", index=pivot.index, columns=pivot.columns)

    presampler_cols = [c for c in pivot.columns if c[0] != "baseline"]
    prev_ps = None
    for col in presampler_cols:
        if prev_ps is not None and col[0] != prev_ps:
            styles[col] = styles[col].apply(lambda _: "border-left: 3px solid #555")
        prev_ps = col[0]

    for i, idx in enumerate(pivot.index):
        ctrl = idx[0]
        stds = idx[1]
        if i > 0:
            prev_ctrl = pivot.index[i - 1][0]
            prev_stds = pivot.index[i - 1][1]
            if ctrl != prev_ctrl:
                for col in pivot.columns:
                    existing = styles.iloc[i][col]
                    sep = "border-top: 3px solid #555"
                    styles.iloc[i, styles.columns.get_loc(col)] = f"{existing}; {sep}" if existing else sep
            elif stds != prev_stds:
                for col in pivot.columns:
                    existing = styles.iloc[i][col]
                    sep = "border-top: 2px solid #999"
                    styles.iloc[i, styles.columns.get_loc(col)] = f"{existing}; {sep}" if existing else sep
    return styles


def build_delta_table(group_df, dynamics, horizon):
    sr_pivot = group_df.pivot_table(
        index=["controller", "stds", "n_samples"],
        columns=["presampler", "n_presamples"],
        values="total_sr",
        aggfunc="first",
    )
    delta_pivot = group_df.pivot_table(
        index=["controller", "stds", "n_samples"],
        columns=["presampler", "n_presamples"],
        values="delta_pct",
        aggfunc="first",
    )

    col_order = pd.MultiIndex.from_product(
        [PRESAMPLER_ORDER, PRESAMPLES_ORDER],
        names=["presampler", "n_presamples"],
    )
    sr_pivot = sr_pivot.reindex(columns=col_order)
    delta_pivot = delta_pivot.reindex(columns=col_order)

    display_df = sr_pivot.copy().astype(object)
    for col in col_order:
        display_df[col] = [
            _format_cell(sr, delta)
            for sr, delta in zip(sr_pivot[col], delta_pivot[col])
        ]

    baseline_vals = (
        group_df.drop_duplicates(subset=["controller", "stds", "n_samples"])
        .set_index(["controller", "stds", "n_samples"])["baseline_sr"]
    )
    display_df.insert(0, ("baseline", "SR"), baseline_vals.apply(lambda v: f"{v:.3f}"))

    idx_order = pd.MultiIndex.from_product(
        [CONTROLLER_ORDER, ["low", "high"], sorted(group_df["n_samples"].unique())],
        names=["controller", "stds", "n_samples"],
    )
    display_df = display_df.reindex(idx_order).dropna(how="all")
    display_df.index.names = ["Controller", "Stds", "Samples"]

    presampler_cols = [c for c in display_df.columns if c[0] != "baseline"]

    styled = (
        display_df.style
        .map(_color_delta, subset=presampler_cols)
        .apply(lambda _: _border_styles(display_df), axis=None)
        .set_caption(f"{dynamics.capitalize()} — Horizon {horizon}")
        .set_table_styles([
            {"selector": "caption", "props": [
                ("font-size", "16px"), ("font-weight", "bold"), ("margin-bottom", "8px"),
            ]},
            {"selector": "th", "props": [
                ("font-size", "12px"), ("text-align", "center"), ("padding", "6px 10px"),
            ]},
            {"selector": "td", "props": [
                ("text-align", "center"), ("padding", "5px 8px"), ("font-size", "12px"),
                ("white-space", "nowrap"),
            ]},
            {"selector": "table", "props": [
                ("border-collapse", "collapse"), ("margin-bottom", "24px"),
            ]},
            {"selector": "", "props": [("border", "1px solid #ccc")]},
        ])
    )
    return styled


for dynamics in ["unicycle", "bicycle"]:
    for horizon in [16, 26]:
        group = variants[(variants["dynamics"] == dynamics) & (variants["horizon"] == horizon)]
        if group.empty:
            continue
        styled_table = build_delta_table(group, dynamics, horizon)
        display(styled_table)

### Latex export

In [ ]:
PRESAMPLER_LATEX = {"nf": "NF", "fm": "FM", "nf_steer": "NF-Steer", "fm_steer": "FM-Steer"}
CONTROLLER_LATEX = {"mppi": "MPPI", "logmppi": "LogMPPI"}

PRESAMPLER_HALVES = [
    (["nf", "fm"], "a"),
    (["nf_steer", "fm_steer"], "b"),
]


def _latex_cell(sr, delta):
    if pd.isna(sr) or pd.isna(delta):
        return "---"
    intensity = min(abs(delta) / 20, 1.0)
    color_pct = int(10 + 35 * intensity)
    if delta > 0:
        return f"\\cellcolor{{posgreen!{color_pct}}}\\textcolor{{posgreentext}}{{{sr:.3f} ({delta:+.1f}\\%)}}"
    if delta < 0:
        return f"\\cellcolor{{negred!{color_pct}}}\\textcolor{{negredtext}}{{{sr:.3f} ({delta:+.1f}\\%)}}"
    return f"{sr:.3f} ({delta:+.1f}\\%)"


def build_latex_table(group_df, dynamics, horizon, ps_subset, label_suffix):
    sr_pivot = group_df.pivot_table(
        index=["controller", "stds", "n_samples"],
        columns=["presampler", "n_presamples"],
        values="total_sr",
        aggfunc="first",
    )
    delta_pivot = group_df.pivot_table(
        index=["controller", "stds", "n_samples"],
        columns=["presampler", "n_presamples"],
        values="delta_pct",
        aggfunc="first",
    )
    col_order = pd.MultiIndex.from_product([ps_subset, PRESAMPLES_ORDER])
    sr_pivot = sr_pivot.reindex(columns=col_order)
    delta_pivot = delta_pivot.reindex(columns=col_order)

    baseline_vals = (
        group_df.drop_duplicates(subset=["controller", "stds", "n_samples"])
        .set_index(["controller", "stds", "n_samples"])["baseline_sr"]
    )
    idx_order = pd.MultiIndex.from_product(
        [CONTROLLER_ORDER, ["low", "high"], sorted(group_df["n_samples"].unique())]
    )
    sr_pivot = sr_pivot.reindex(idx_order).dropna(how="all")
    delta_pivot = delta_pivot.reindex(idx_order).dropna(how="all")
    baseline_vals = baseline_vals.reindex(idx_order).dropna()

    n_ps = len(ps_subset)
    n_nps = len(PRESAMPLES_ORDER)
    total_data_cols = 4 + n_ps * n_nps

    samples_list = sorted(group_df["n_samples"].unique())
    rows_per_stds = len(samples_list)
    rows_per_ctrl = 2 * rows_per_stds

    col_spec = "ll" + "r|" + "C|" + "|".join(["C" * n_nps] * n_ps)

    ps_names = " \\& ".join(PRESAMPLER_LATEX[p] for p in ps_subset)
    caption = f"{dynamics.capitalize()}, $H = {horizon}$ --- {ps_names} pre-samplers"

    lines = []
    lines.append("\\begin{table}[H]")
    lines.append(f"\\caption{{{caption}}}")
    lines.append(f"\\label{{tab:sr_{dynamics}_h{horizon}_{label_suffix}}}")
    lines.append("\\begin{adjustwidth}{-\\extralength}{0cm}")
    lines.append(f"\\begin{{tabularx}}{{\\fulllength}}{{{col_spec}}}")
    lines.append("\\toprule")

    header1_parts = ["", "", "", ""]
    for i, ps in enumerate(ps_subset):
        sep = "" if i == n_ps - 1 else "|"
        header1_parts.append(f"\\multicolumn{{{n_nps}}}{{c{sep}}}{{{PRESAMPLER_LATEX[ps]}}}")
    lines.append(" & ".join(header1_parts) + " \\\\")

    data_col_start = 5
    for i in range(n_ps):
        s = data_col_start + i * n_nps
        e = s + n_nps - 1
        lines.append(f"\\cmidrule(lr){{{s}-{e}}}")

    header2_parts = ["Controller", "Stds", "Samples", "Baseline"]
    for ps in ps_subset:
        for nps in PRESAMPLES_ORDER:
            header2_parts.append(str(nps))
    lines.append(" & ".join(header2_parts) + " \\\\")
    lines.append("\\midrule")

    prev_ctrl = None
    prev_stds = None
    for idx in sr_pivot.index:
        ctrl, stds, n_samp = idx

        if prev_ctrl is not None and ctrl != prev_ctrl:
            lines.append("\\midrule")
            prev_stds = None
        elif prev_stds is not None and stds != prev_stds:
            lines.append(f"\\cmidrule(lr){{2-{total_data_cols}}}")

        if ctrl != prev_ctrl:
            ctrl_label = f"\\multirow{{{rows_per_ctrl}}}{{*}}{{\\textbf{{{CONTROLLER_LATEX[ctrl]}}}}}"
        else:
            ctrl_label = ""

        if stds != prev_stds or ctrl != prev_ctrl:
            stds_label = f"\\multirow{{{rows_per_stds}}}{{*}}{{{stds}}}"
        else:
            stds_label = ""

        parts = [ctrl_label, stds_label, str(int(n_samp)), f"{baseline_vals[idx]:.3f}"]
        for col in col_order:
            parts.append(_latex_cell(sr_pivot.loc[idx, col], delta_pivot.loc[idx, col]))

        lines.append(" & ".join(parts) + " \\\\")
        prev_ctrl = ctrl
        prev_stds = stds

    lines.append("\\bottomrule")
    lines.append("\\end{tabularx}")
    lines.append("\\end{adjustwidth}")
    lines.append("\\end{table}")
    return "\n".join(lines)


preamble = r"""\definecolor{posgreen}{RGB}{76, 175, 80}
\definecolor{posgreentext}{RGB}{27, 94, 32}
\definecolor{negred}{RGB}{244, 67, 54}
\definecolor{negredtext}{RGB}{183, 28, 28}
"""
print(preamble)

for dynamics in ["unicycle", "bicycle"]:
    for horizon in [16, 26]:
        group = variants[(variants["dynamics"] == dynamics) & (variants["horizon"] == horizon)]
        if group.empty:
            continue
        for ps_subset, label_suffix in PRESAMPLER_HALVES:
            latex_src = build_latex_table(group, dynamics, horizon, ps_subset, label_suffix)
            print(latex_src)
            print()

### Summary of the advantages

In [ ]:
from IPython.display import HTML

# ── Headline statistics ──────────────────────────────────────────────────────
n_base = variants.groupby(BASE_KEY_COLS).ngroups
best_delta_per_base = variants.groupby(BASE_KEY_COLS)["delta_pct"].max()
n_improved_by_best = (best_delta_per_base > 0).sum()
overall_win_rate = (variants["delta_pct"] > 0).mean() * 100
avg_improvement_when_positive = variants.loc[variants["delta_pct"] > 0, "delta_pct"].mean()

display(HTML(f"""
<div style="display:flex; gap:32px; margin:12px 0 20px 0; flex-wrap:wrap;">
  <div style="background:#e8f5e9; border-left:4px solid #4caf50; padding:12px 20px; border-radius:4px; min-width:180px;">
    <div style="font-size:13px; color:#555;">Configs improved<br>(best pre-sampler)</div>
    <div style="font-size:28px; font-weight:bold; color:#2e7d32;">{n_improved_by_best}/{n_base} ({n_improved_by_best/n_base*100:.0f}%)</div>
  </div>
  <div style="background:#e8f5e9; border-left:4px solid #4caf50; padding:12px 20px; border-radius:4px; min-width:180px;">
    <div style="font-size:13px; color:#555;">Overall variant<br>win rate</div>
    <div style="font-size:28px; font-weight:bold; color:#2e7d32;">{overall_win_rate:.0f}%</div>
  </div>
  <div style="background:#e8f5e9; border-left:4px solid #4caf50; padding:12px 20px; border-radius:4px; min-width:180px;">
    <div style="font-size:13px; color:#555;">Avg improvement<br>(when positive)</div>
    <div style="font-size:28px; font-weight:bold; color:#2e7d32;">+{avg_improvement_when_positive:.1f}%</div>
  </div>
  <div style="background:#e8f5e9; border-left:4px solid #4caf50; padding:12px 20px; border-radius:4px; min-width:180px;">
    <div style="font-size:13px; color:#555;">Best single<br>improvement</div>
    <div style="font-size:28px; font-weight:bold; color:#2e7d32;">+{variants['delta_pct'].max():.0f}%</div>
  </div>
</div>
"""))

# ── Figure 1: Win rate & average improvement per presampler ──────────────────
PRESAMPLER_LABELS = {"nf": "NF", "fm": "FM", "nf_steer": "NF-Steer", "fm_steer": "FM-Steer"}
PS_COLORS = {500: "#90caf9", 1000: "#42a5f5", 2500: "#1565c0"}

summary = variants.groupby(["presampler", "n_presamples"]).agg(
    win_rate=("delta_pct", lambda x: (x > 0).mean() * 100),
    mean_improvement=("delta_pct", "mean"),
).reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
bar_width = 0.22
x = np.arange(len(PRESAMPLER_ORDER))

for i, nps in enumerate(PRESAMPLES_ORDER):
    sub = summary[summary["n_presamples"] == nps].set_index("presampler").reindex(PRESAMPLER_ORDER)
    offset = (i - 1) * bar_width
    ax1.bar(x + offset, sub["win_rate"], bar_width, label=f"{nps}", color=PS_COLORS[nps], edgecolor="white")
    ax2.bar(x + offset, sub["mean_improvement"], bar_width, label=f"{nps}", color=PS_COLORS[nps], edgecolor="white")

for ax, ylabel, title in [
    (ax1, "Configs improved (%)", "Win Rate by Pre-sampler"),
    (ax2, "Mean relative improvement (%)", "Average SR Improvement by Pre-sampler"),
]:
    ax.set_xticks(x)
    ax.set_xticklabels([PRESAMPLER_LABELS[p] for p in PRESAMPLER_ORDER], fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=13, fontweight="bold", pad=10)
    ax.legend(title="Pre-samples", fontsize=9, title_fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)

ax1.set_ylim(0, 105)
ax2.axhline(0, color="#888", linewidth=0.8, linestyle="--")

fig.tight_layout()
plt.show()

# ── Figure 2: Baseline vs Best pre-sampler (paired dot plot, faceted) ────────
fig2, axes2 = plt.subplots(2, 2, figsize=(14, 9), sharey=False)

for (dynamics, horizon), ax in zip(
    [("unicycle", 16), ("unicycle", 26), ("bicycle", 16), ("bicycle", 26)],
    axes2.flat,
):
    grp = variants[(variants["dynamics"] == dynamics) & (variants["horizon"] == horizon)]
    if grp.empty:
        ax.set_visible(False)
        continue

    best_idx = grp.groupby(["controller", "stds", "n_samples"])["delta_pct"].idxmax()
    best = grp.loc[best_idx].copy()
    best["label"] = best.apply(
        lambda r: f"{r['controller']} | {r['stds']} | {r['n_samples']:.0f}", axis=1
    )
    best = best.sort_values("baseline_sr")
    best = best.reset_index(drop=True)

    y = np.arange(len(best))

    for i, row in best.iterrows():
        color = "#4caf50" if row["delta_pct"] > 0 else "#f44336"
        ax.plot(
            [row["baseline_sr"], row["total_sr"]], [y[i], y[i]],
            color=color, linewidth=2, alpha=0.7, zorder=1,
        )
    ax.scatter(best["baseline_sr"], y, color="#9e9e9e", s=50, zorder=2, label="Baseline")
    ax.scatter(best["total_sr"], y, color="#2e7d32", s=50, zorder=2, label="Best pre-sampler")

    ax.set_yticks(y)
    ax.set_yticklabels(best["label"], fontsize=9)
    ax.set_xlabel("Total Success Rate", fontsize=10)
    ax.set_title(f"{dynamics.capitalize()} — Horizon {horizon}", fontsize=12, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.3)
    ax.legend(fontsize=9, loc="lower right")

fig2.suptitle(
    "Every base controller is improved by its best pre-sampler",
    fontsize=14, fontweight="bold", y=1.01,
)
fig2.tight_layout()
plt.show()

# Analysis for Sampling-Only controllers

In [ ]:
samp_rows = []
for r in RESULTS_SAMPLING_ONLY:
    samp_rows.append({
        "dynamics": r.dynamics,
        "horizon": r.horizon,
        "presampler": r.presampler_type,
        "n_presamples": r.n_presamples,
        "total_sr": r.result_metrics.sr.total,
    })
df_samp = pd.DataFrame(samp_rows)

PRESAMPLER_LABELS = {"nf": "NF", "fm": "FM", "nf_steer": "NF-Steer", "fm_steer": "FM-Steer"}


def build_sampling_table(group_df, dynamics, horizon):
    pivot = group_df.pivot_table(
        index="presampler",
        columns="n_presamples",
        values="total_sr",
        aggfunc="first",
    )
    pivot = pivot.reindex(index=PRESAMPLER_ORDER, columns=PRESAMPLES_ORDER)
    pivot.index = pivot.index.map(PRESAMPLER_LABELS)
    pivot.index.name = "Pre-sampler"
    pivot.columns.name = "Pre-samples"

    styled = (
        pivot.style
        .format("{:.3f}")
        .background_gradient(cmap="Greens", axis=None)
        .set_caption(f"{dynamics.capitalize()} — Horizon {horizon}")
        .set_table_styles([
            {"selector": "caption", "props": [
                ("font-size", "16px"), ("font-weight", "bold"), ("margin-bottom", "8px"),
            ]},
            {"selector": "th", "props": [
                ("font-size", "12px"), ("text-align", "center"), ("padding", "6px 12px"),
            ]},
            {"selector": "td", "props": [
                ("text-align", "center"), ("padding", "6px 12px"), ("font-size", "12px"),
            ]},
            {"selector": "table", "props": [
                ("border-collapse", "collapse"), ("margin-bottom", "24px"),
            ]},
            {"selector": "", "props": [("border", "1px solid #ccc")]},
        ])
    )
    return styled


for dynamics in ["unicycle", "bicycle"]:
    for horizon in [16, 26]:
        group = df_samp[(df_samp["dynamics"] == dynamics) & (df_samp["horizon"] == horizon)]
        if group.empty:
            continue
        display(build_sampling_table(group, dynamics, horizon))

In [ ]:
def build_sampling_latex_table(group_df, dynamics, horizon):
    pivot = group_df.pivot_table(
        index="presampler",
        columns="n_presamples",
        values="total_sr",
        aggfunc="first",
    )
    pivot = pivot.reindex(index=PRESAMPLER_ORDER, columns=PRESAMPLES_ORDER)

    n_nps = len(PRESAMPLES_ORDER)
    col_spec = "l|" + "C" * n_nps

    lines = []
    lines.append("\\begin{table}[H]")
    lines.append(f"\\caption{{Sampling-only controller, {dynamics.capitalize()}, $H = {horizon}$}}")
    lines.append(f"\\label{{tab:sr_sampling_{dynamics}_h{horizon}}}")
    lines.append("\\begin{adjustwidth}{-\\extralength}{0cm}")
    lines.append(f"\\begin{{tabularx}}{{\\fulllength}}{{{col_spec}}}")
    lines.append("\\toprule")

    header_parts = ["Pre-sampler"] + [str(nps) for nps in PRESAMPLES_ORDER]
    lines.append(" & ".join(header_parts) + " \\\\")
    lines.append("\\midrule")

    for ps in PRESAMPLER_ORDER:
        parts = [PRESAMPLER_LATEX[ps]]
        for nps in PRESAMPLES_ORDER:
            val = pivot.loc[ps, nps]
            parts.append(f"{val:.3f}" if not pd.isna(val) else "---")
        lines.append(" & ".join(parts) + " \\\\")

    lines.append("\\bottomrule")
    lines.append("\\end{tabularx}")
    lines.append("\\end{adjustwidth}")
    lines.append("\\end{table}")
    return "\n".join(lines)


for dynamics in ["unicycle", "bicycle"]:
    for horizon in [16, 26]:
        group = df_samp[(df_samp["dynamics"] == dynamics) & (df_samp["horizon"] == horizon)]
        if group.empty:
            continue
        print(build_sampling_latex_table(group, dynamics, horizon))
        print()

### vs MPPI controllers 

In [ ]:
def _describe_config(row, category):
    if category == "sampling":
        return f"{PRESAMPLER_LABELS[row['presampler']]}, {int(row['n_presamples'])} pre-samples"
    ctrl = row["controller"].upper() if row["controller"] == "mppi" else "LogMPPI"
    base = f"{ctrl}, {row['stds']} stds, {int(row['n_samples'])} samples"
    if category == "with_ps":
        base += f" + {PRESAMPLER_LABELS[row['presampler']]}, {int(row['n_presamples'])} pre-samples"
    return base


def build_comparison(dynamics, horizon):
    # 1) Best sampling-only
    s_grp = df_samp[(df_samp["dynamics"] == dynamics) & (df_samp["horizon"] == horizon)]
    best_samp = s_grp.loc[s_grp["total_sr"].idxmax()]

    # 2) Best baseline MPPI/LogMPPI (no presampler)
    base_grp = df[(df["dynamics"] == dynamics) & (df["horizon"] == horizon) & (df["presampler"] == "none")]
    best_base = base_grp.loc[base_grp["total_sr"].idxmax()]

    # 3) Best MPPI/LogMPPI with presampler
    ps_grp = variants[(variants["dynamics"] == dynamics) & (variants["horizon"] == horizon)]
    best_ps = ps_grp.loc[ps_grp["total_sr"].idxmax()]

    rows = [
        {"Category": "Sampling-only", "Configuration": _describe_config(best_samp, "sampling"),
         "Total SR": best_samp["total_sr"]},
        {"Category": "Best baseline", "Configuration": _describe_config(best_base, "baseline"),
         "Total SR": best_base["total_sr"]},
        {"Category": "Best with pre-sampler", "Configuration": _describe_config(best_ps, "with_ps"),
         "Total SR": best_ps["total_sr"]},
    ]
    return pd.DataFrame(rows)


def _highlight_best(s):
    is_max = s == s.max()
    return ["font-weight: bold; background-color: rgba(76, 175, 80, 0.25); color: #1b5e20" if v else "" for v in is_max]


for dynamics in ["unicycle", "bicycle"]:
    for horizon in [16, 26]:
        comp_df = build_comparison(dynamics, horizon)
        styled = (
            comp_df.style
            .apply(_highlight_best, subset=["Total SR"])
            .format({"Total SR": "{:.3f}"})
            .hide(axis="index")
            .set_caption(f"{dynamics.capitalize()} — Horizon {horizon}")
            .set_table_styles([
                {"selector": "caption", "props": [
                    ("font-size", "16px"), ("font-weight", "bold"), ("margin-bottom", "8px"),
                ]},
                {"selector": "th", "props": [
                    ("font-size", "13px"), ("text-align", "center"), ("padding", "8px 14px"),
                    ("background-color", "#f5f5f5"),
                ]},
                {"selector": "td", "props": [
                    ("text-align", "center"), ("padding", "8px 14px"), ("font-size", "12px"),
                ]},
                {"selector": "td:nth-child(2)", "props": [("text-align", "left")]},
                {"selector": "table", "props": [
                    ("border-collapse", "collapse"), ("margin-bottom", "24px"), ("min-width", "500px"),
                ]},
                {"selector": "", "props": [("border", "1px solid #ccc")]},
            ])
        )
        display(styled)

In [ ]:
def build_comparison_latex(dynamics, horizon):
    comp_df = build_comparison(dynamics, horizon)
    best_idx = comp_df["Total SR"].idxmax()

    lines = []
    lines.append("\\begin{table}[H]")
    lines.append(f"\\caption{{Best controllers comparison, {dynamics.capitalize()}, $H = {horizon}$}}")
    lines.append(f"\\label{{tab:comparison_{dynamics}_h{horizon}}}")
    lines.append("\\begin{adjustwidth}{-\\extralength}{0cm}")
    lines.append("\\begin{tabularx}{\\fulllength}{l|l|C}")
    lines.append("\\toprule")
    lines.append("Category & Configuration & Total SR \\\\")
    lines.append("\\midrule")

    for i, row in comp_df.iterrows():
        sr_str = f"{row['Total SR']:.3f}"
        if i == best_idx:
            sr_str = f"\\textbf{{{sr_str}}}"
        config_str = row["Configuration"].replace("&", "\\&")
        lines.append(f"{row['Category']} & {config_str} & {sr_str} \\\\")

    lines.append("\\bottomrule")
    lines.append("\\end{tabularx}")
    lines.append("\\end{adjustwidth}")
    lines.append("\\end{table}")
    return "\n".join(lines)


for dynamics in ["unicycle", "bicycle"]:
    for horizon in [16, 26]:
        print(build_comparison_latex(dynamics, horizon))
        print()